### Create Tokens

In [1]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
  raw_text = f.read()

print("Total num of characters:", len(raw_text))
print(raw_text[:99])

Total num of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
import re

def tokenize_story(raw_text):
    tokens = re.split(r'([,.:;?!()"\'\-]|\s+)', raw_text)
    tokens = [t.strip() for t in tokens if t.strip()]
    return tokens


In [5]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [6]:
print(len(preprocessed))

4690


### Create Token ID's

In [8]:
## Sort and remove repetative words
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"{len(all_words)} unique words")

1130 unique words


In [9]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [10]:
for i,item in enumerate(vocab.items()):
  print(item)
  if i>=50:
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


### Encode -- split and remove and then return token id and decode is reverse

In [11]:
import re

class SimpleTokenizerV1:

    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def _tokenize(self, text):
        # Split on punctuation + whitespace
        tokens = re.split(r'(--|[,.:;?!()"\'\-]|\s+)', text)
        tokens = [t.strip() for t in tokens if t.strip()]
        return tokens

    def encode(self, text):
        tokens = self._tokenize(text)

        # Handle unknown tokens
        ids = [
            self.str_to_int[token]
            if token in self.str_to_int
            else self.str_to_int["<UNK>"]
            for token in tokens
        ]

        return ids

    def decode(self, ids):
        tokens = [self.int_to_str[i] for i in ids]

        text = " ".join(tokens)

        # Remove unwanted spaces before punctuation
        text = re.sub(r"\s+([,.!?;:])", r"\1", text)

        return text

In [14]:
tokenizer = SimpleTokenizerV1(vocab)

text = """It's the last he painted"""
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[56, 2, 850, 988, 602, 533, 746]
It ' s the last he painted


### OOV problem

In [15]:
text = """ iam sangu"""
ids = tokenizer.encode(text)

KeyError: '<UNK>'

### Added special work tokenizers

In [18]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<UNK>","<SOS>","<EOS>"])
vocab = {token:integer for integer , token in enumerate(all_tokens) }

In [19]:
len(vocab.items())

1133

In [20]:
import re

class SimpleTokenizerV1:

    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def _tokenize(self, text):
        tokens = re.split(r'(--|[,.:;?!()"\'\-]|\s+)', text)
        tokens = [t.strip() for t in tokens if t.strip()]
        return tokens

    def encode(self, text):
        tokens = self._tokenize(text)

        ids = [
            self.str_to_int[token]
            if token in self.str_to_int
            else self.str_to_int["<UNK>"]
            for token in tokens
        ]

        # Add special tokens
        ids = (
            [self.str_to_int["<SOS>"]]
            + ids
            + [self.str_to_int["<EOS>"]]
        )

        return ids

    def decode(self, ids):
        tokens = [self.int_to_str[i] for i in ids]

        text = " ".join(tokens)

        # Remove spaces before punctuation
        text = re.sub(r"\s+([,.!?;:])", r"\1", text)

        return text

In [21]:
tokenizer = SimpleTokenizerV1(vocab)

text = """It's the last he painted"""
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1131, 56, 2, 850, 988, 602, 533, 746, 1132]
<SOS> It ' s the last he painted <EOS>


In [24]:
text = """ iam sangu"""
ids = tokenizer.encode(text)
print(ids)
decode = tokenizer.decode(ids)
print(decode)

[1131, 1130, 1130, 1132]
<SOS> <UNK> <UNK> <EOS>


### Byte Pair Encoding

In [25]:
!pip install tiktoken
import tiktoken
import importlib

In [26]:
tokenizer = tiktoken.get_encoding("gpt2")

In [28]:
text = (
    "Hello, do you like tea? <|endoftext|>"
)
tokens = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(tokens)
decode = tokenizer.decode(tokens)
print(decode)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256]
Hello, do you like tea? <|endoftext|>


### create i/p o/p pairs

In [29]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [30]:
enc_sample = enc_text[50:]

In [31]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [32]:
for i in range(1,context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(f"x: {context}")
  print(f"y: {desired}")

  print(context, "---->", desired)

x: [290]
y: 4920
[290] ----> 4920
x: [290, 4920]
y: 2241
[290, 4920] ----> 2241
x: [290, 4920, 2241]
y: 287
[290, 4920, 2241] ----> 287
x: [290, 4920, 2241, 287]
y: 257
[290, 4920, 2241, 287] ----> 257


In [33]:
for i in range(1,context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(tokenizer.decode(context), "---->" , tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


### Data Loader

In [34]:
import torch

In [35]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self,txt,tokenizer,max_length,stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    for i in range(0,len(token_ids) - max_length,stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self,idx):
    return self.input_ids[idx], self.target_ids[idx]

In [36]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)

  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)
  return dataloader

In [37]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False,
    num_workers=0
)

In [39]:
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [40]:
vocab_size = 50257
output_dim = 256

In [41]:
token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [42]:
 max_length = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=max_length,
    stride=max_length,
    shuffle=False
)

data_iter = iter(dataloader)
inputs,target = next(data_iter)

In [ ]:
token_embeddings = token_embedding(inputs)
print(token_embeddings.shape)

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)